In [ ]:
"""
Author    : Mohammad Norizadeh Cherloo
Project   : Word Generation
Algorithm : Long Short-Term Memory (LSTM) – PyTorch
"""
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import re
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
import random
import os
import string

In [35]:
## Reading and processing text
with open('dataset.txt', 'r', encoding="utf8") as fp:
    raw=fp.read()

print('Total Length (characters):', len(raw))

Total Length (characters): 2751174


**Tokenization**

In [36]:
def tokenize(doc):
    splited_doc= doc.split()
    splited_tokens= []
    for word in splited_doc:
        splited_tokens.extend(word.replace('.', ' .').split())
    tokens= [token.lower() for token in splited_tokens]
    tokens= [token for token in tokens if token.isalpha() or token== '.' ]
    return tokens

doc_tokens= tokenize(raw)
print(len(doc_tokens))
# doc_tokens[:20]

584955


**Create vocabulary**

In [37]:
vocabulary= sorted(set(doc_tokens))
print(len(vocabulary))
word2indx= {word:index for index,word in enumerate(vocabulary)}
indx2word= {index:word for index,word in enumerate(vocabulary)}
# vocabulary[:20]
# word2indx
# indx2word

5371


ِ**Define sequence to index function**

In [38]:
def sequence_to_index(sequence,word2indx):
    lst=[]
    for token in sequence:
        lst.append(word2indx[token])
        
    seq_index= torch.tensor(lst).long()
    return seq_index

sequence= doc_tokens[:50]
seq_index=sequence_to_index(sequence,word2indx)
# print(seq_index)

**Define Dataset**

In [39]:
class TextDataset(Dataset):
    def __init__(self,doc_tokens,word2indx,max_len=50):
        super().__init__()
        self.data=[]
        self.target=[]
        
        doc_tokens_indx= sequence_to_index(doc_tokens,word2indx)
        indices= torch.arange(len(doc_tokens))
        # a=2
        for i in range(len(doc_tokens_indx)-(max_len+1)):
            pos= indices[i:i+max_len+1]
            chunk= doc_tokens_indx[pos]
            input= chunk[:-1]
            output= chunk[1:]
            self.data.append(input)
            self.target.append(output)
        
    def  __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        x= self.data[index] 
        y= self.target[index] 
        return x,y

In [40]:
dataset= TextDataset(doc_tokens,word2indx,max_len=50)
# print(doc_tokens[:20])
# print(dataset.doc_tokens_indx[:20])

In [41]:
# len(dataset
# print(len(dataset))
# print(len(doc_tokens))

In [42]:
x,y=dataset[0]
# print(x)
# print(y)

**Define DataLoader**

In [43]:
train_indx,valid_indx= train_test_split(list(range(len(dataset))),
                                        test_size=0.15,
                                        random_state=42)

train_subset= torch.utils.data.Subset(dataset,train_indx)
valid_subset= torch.utils.data.Subset(dataset,valid_indx)

BATCH_SIZE=32
train_loader= DataLoader(train_subset,batch_size=BATCH_SIZE,
                         shuffle=True,
                         drop_last=True)

valid_loader= DataLoader(valid_subset,batch_size=BATCH_SIZE,
                         shuffle=False,
                         drop_last=True)


**Define RNN model**

In [44]:
class WordgenRNN(nn.Module):
    def __init__(self, vocab_size,
                 embed_dim=64,
                 hidden_dim=128, 
                 n_layers=1,
                 bidirectional=False,
                 dropout=0.3):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(input_size=embed_dim,
                            hidden_size=hidden_dim,
                            num_layers=n_layers,
                            batch_first=True,
                            bidirectional=bidirectional,
                            dropout=dropout if n_layers>1 else 0.0)
        
        self.fc1 = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None,cell=None):
        if hidden is None or cell is None:
            h_prev, c_prev= self.init_zero_state(batch_size=x.size(0))
        else:
            h_prev=hidden
            c_prev=cell
           
        embedded = self.embed(x)# (B, n_seq, embed_dim)
        out, (hidden,cell) = self.lstm(embedded,(h_prev,c_prev)) # out: (B, n_seq, hidden)
        output = self.fc1(out)
        return output, hidden,cell
    
    def init_zero_state(self, batch_size):
        # returns tuple of zeros for (h0, c0) with shape (n_layers, batch, hidden_dim)
        num_layers = self.lstm.num_layers
        hidden_size = self.lstm.hidden_size
        
        hidden0 = torch.zeros(num_layers, batch_size, hidden_size)
        cell0 = torch.zeros(num_layers, batch_size, hidden_size)  
        return hidden0,cell0
        

**text generation function**

In [97]:
def generate_text(model,prime_str='the lord of',
                  word2indx=word2indx,
                  indx2word=indx2word,
                  seq_length=20,
                  temperature=0.8):
    
    model.eval()
    tokens= tokenize(prime_str)
    prime_index= sequence_to_index(tokens,word2indx)
    # init h0,c0
    hidden,cell_state= model.init_zero_state(batch_size=1)
    with torch.no_grad():
        for i in range(len(tokens)-1):
            inp= prime_index[i].view(1,-1) # b*1
            out,hidden,cell_state= model(inp,hidden,cell_state)
        
        generated_text= prime_str
        inp= prime_index[-1].view(1,-1)
        for i in range(seq_length):
            logits,hidden,cell_state= model(inp,hidden,cell_state)
            logits= logits.view(1,-1)
            probs= torch.softmax(logits/(temperature+0.00000001),dim=1)
            # indx= torch.argmax(probs)
            indx= torch.multinomial(probs,1)
            next_word= indx2word[int(indx)]
            generated_text+=' '+next_word
            
            inp= sequence_to_index([next_word],word2indx).view(1,-1)
            
        model.train()
    return generated_text.replace(" . ",". ")


def generate_text2(model,prime_str='the lord of',
                  word2indx=word2indx,
                  indx2word=indx2word,
                  seq_length=20,
                  temperature=0.8):
    model.eval()
    # prepare start text
    prime_token= tokenize(prime_str)
    prime_index= sequence_to_index(prime_token,word2indx)
    hidden,cell_state= model.init_zero_state(batch_size=1)
    for i in range(len(prime_token)-1):
        inp= prime_index[i].view(1,-1)
        out,hidden,cell_state= model(inp,hidden,cell_state)
    inp= prime_index[-1].view(1,-1)
    
    generated_text= prime_str
    for i in range(seq_length):
        logits,hidden,cell_state= model(inp,hidden,cell_state)
        logits= logits.view(-1)
        index= top_p_sampling(logits.detach(), temperature, top_p=0.9)
        next_word= indx2word[index]
        generated_text+= ' '+ next_word
        inp= sequence_to_index([next_word],word2indx).view(1,-1)
    # print(generated_test)
    model.train()
    return generated_text.replace(" . ", ". ")



def top_p_sampling(logits, temperature=1.0, top_p=0.9):

    # Apply temperature scaling
    scaled_logits = logits / temperature

    # Convert logits to probabilities using softmax
    probabilities = torch.softmax(scaled_logits, dim=-1)

    # Sort probabilities and compute cumulative sum
    sorted_indices = torch.argsort(probabilities, descending=True)
    sorted_probabilities = probabilities[sorted_indices]
    cumulative_probabilities = torch.cumsum(sorted_probabilities, dim=-1)

    # Apply top-p filtering
    indices_to_keep = cumulative_probabilities <= top_p
    truncated_probabilities = sorted_probabilities[indices_to_keep]

    # Rescale the probabilities
    truncated_probabilities /= torch.sum(truncated_probabilities)

    # Convert to numpy arrays for random choice
    truncated_probabilities = truncated_probabilities.cpu().numpy()
    sorted_indices = sorted_indices.cpu().numpy()
    indices_to_keep = indices_to_keep.cpu().numpy()

    # Sample from the truncated distribution
    if not indices_to_keep.any():
        # Handle the empty case - for example, using regular sampling without top-p
        probabilities = torch.softmax(logits / temperature, dim=-1)
        next_word_index = torch.multinomial(probabilities, 1).item()
    else:
        # Existing sampling process
        next_word_index = np.random.choice(sorted_indices[indices_to_keep], p=truncated_probabilities)
    return next_word_index

In [101]:
vocab_size=len(vocabulary)
embed_dim = 256
rnn_hidden_size = 512

model = WordgenRNN(vocab_size=vocab_size,
                 embed_dim=embed_dim, 
                 hidden_dim=rnn_hidden_size,
                 n_layers=2,
                 bidirectional=False, 
                 dropout=0.3)

optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()   # applied per time step (flattened)

In [102]:
generated_text1= generate_text(model,prime_str='the lord of',
                              word2indx=word2indx,
                              indx2word=indx2word,
                              seq_length=20,
                              temperature=0.8)

generated_text2= generate_text2(model,prime_str='the lord of',
                              word2indx=word2indx,
                              indx2word=indx2word,
                              seq_length=20,
                              temperature=0.8)

print(generated_text1)
print(generated_text2)

the lord of elegant burned right exactly spent perform knocked poured wing worked waved all borne wolves reeds engaged resemblance getting wealth appearance
the lord of corridor swimming damp somebody islands until safely finns bitter artist cleaned trees birth gentle wendys issued knock rage much honey


In [103]:
n_epochs = 5
print_every=20
train_losses = []
val_losses = []
global_step = 0
hidden = None
cell_state=None
for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (xbatch, ybatch,) in enumerate(train_loader):
        optimizer.zero_grad()
        y_pred,_,_= model(xbatch,None,None)
        B,T,V= y_pred.size()
        loss= criterion(y_pred.view(B*T,V),ybatch.view(B*T))
        loss.backward()
        optimizer.step()
        running_loss= loss.item()
        if global_step%print_every ==0:
            print(f'loss in ({epoch}-{global_step}): {running_loss:.4f}')
            # model.eval()
            # generated_text= generate_text(model,
            #                               prime_str='the lord of',
            #                               word2indx=word2indx,
            #                               indx2word=indx2word,
            #                               seq_length=20,
            #                               temperature=0.8)
            # print(generated_text)
            
            generated_text1= generate_text(model,prime_str='the lord of',
                              word2indx=word2indx,
                              indx2word=indx2word,
                              seq_length=20,
                              temperature=0.8)

            generated_text2= generate_text2(model,prime_str='the lord of',
                                        word2indx=word2indx,
                                        indx2word=indx2word,
                                        seq_length=20,
                                        temperature=0.8)

            print(generated_text1)
            print(generated_text2)
            # model.train()
        global_step+=1
        

loss in (0-0): 8.5884
the lord of warned craft evenings devoted curiously quicksilver demis purpose entreated spoon obliged drum lip fancied mustnt delights strongly horner climb colin
the lord of bowl news boat slaves yes capable reflected yere girls cousins crabs mystery swan drawn huntsmans holy woodman charmed faster bunyan
loss in (0-20): 6.3796
the lord of. said a she hear was long nothing the easy i with think so into. his of here all
the lord of they that when to she of. be wouldnt to all then father it had all the theres the of
loss in (0-40): 6.0836
the lord of i i. the see be a throat. and. had therefore all i then their care my changed
the lord of she little. . in tried. now to sea her people and and safely know golden. it of
loss in (0-60): 5.9569
the lord of her couple so. the morning. she shouted on a train and his scarecrow mrs to greatly. there
the lord of my appeared and his day to it in the out. had perhaps in a thought. we the plainly
loss in (0-80): 5.6401
the lord

KeyboardInterrupt: 

In [113]:
generated_text= generate_text(model,prime_str='the girl',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.1)
generated_text.split('.')

['the girl was obliged to appear happy',
 ' she had been in the room',
 ' she was so pretty and cheerful that she did not know what to do',
 ' she was so busy that she could not bear to be reminded her',
 ' but he had not known what she had done',
 ' she was so beautiful that she could not bear to be reminded her',
 ' but he was not persuaded to go out fishing and kept singing on the ground',
 ' bind her for the wood cant you will you wont you will you wont you will you wont']

In [114]:
generated_text= generate_text(model,prime_str='the king',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.1)
generated_text.split('.')

['the king and queen had the most beautiful daughter in the house',
 ' she was terrified by the king and remembered that she was a fairy princess',
 ' she was so beautiful that she could not bear to see her',
 ' she had not been so gentle as before',
 ' she was dressed very simply and her',
 ' she had a great many feelings and they were married that day',
 ' the prince had not been able to move and place for the third time',
 ' the prince was very glad to get out of the house and the heat of the']

In [125]:
generated_text2= generate_text2(model,prime_str='the king',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.6)
generated_text2.split('.')

['the king left him',
 ' there was no sign of the heart to give the rajah to enter the private room',
 ' but the wild rabbits were drowned and each day they journeyed onwards and far away and he was so terribly alone that he jumped down and kissed its shining nose and then seated himself on the floor',
 ' he said',
 ' he was much beloved and his knees immediately made a dash at the best',
 ' he had been placed on the hearth thinking that he was going fishing',
 ' but he was a restless companion and therefore something made']

In [150]:
generated_text2= generate_text2(model,prime_str='the lord of',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.6)
generated_text2.split('.')

['the lord of the house',
 ' the old man was still laughing silent and thoughtful',
 ' it was a horrid sight',
 ' the wonders of the evening',
 ' and yet there was a great deal of confusion on the jury to be alone in danger',
 ' the young officer forgetting was happening that he was expecting something to be so',
 ' and so the two went on',
 ' then the king took a hen and cut it in pieces and sold his cloak',
 ' as he watched the princess the king and queen flung her into the basin with pitch her head and']

In [158]:
generated_text= generate_text(model,prime_str='the lord of',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.1)
generated_text.split('.')

['the lord of the house stairs',
 ' the prince was very glad',
 ' he was a poet',
 ' he was a very happy fellow and he was very angry',
 ' he had a pages dress of terror',
 ' he was a chap who was a general returning or a christmas tree',
 ' he was a very happy man and his mother and his mother had both since it had been sailing past him',
 ' he was a very happy man and he had been unable to think that he was a stranger',
 ' he was a good man and his wife who was']

In [159]:
generated_text= generate_text(model,prime_str='the lord of',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.0)
generated_text.split('.')

['the lord of the house stairs',
 ' the prince was still laughing at rogojin furious',
 ' he had exquisite manners and bowed to the spot where he had been placed for the whole of the battle',
 ' he was a chap of his kind',
 ' he was a good man and his wife who was very glad to see the world',
 ' he was a chap who had been lying asleep and the other one on one side of the river',
 ' he was a very happy man and he had been unable to think that he was a stranger',
 ' he was']

In [172]:
generated_text= generate_text(model,prime_str='i love ',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.0)
generated_text.split('.')

['i love  you to be careful',
 ' we are not afraid of words',
 ' i dont want to die',
 ' ill tell you how to behave',
 ' gretel',
 ' ill taste the porridge for you',
 ' i dont know how to give him to mine',
 ' i dont know how to give him to mine',
 ' i dont know how to give him to mine',
 ' i dont know how to give him to mine',
 ' i dont know how to give him to mine',
 ' i dont know how to give him to mine',
 ' i dont know how to give']

In [183]:
generated_text= generate_text(model,prime_str='i love to',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.1)
generated_text.split('.')

['i love to hunt my husband',
 ' it is a coin which i have described',
 ' we are all desperate here',
 ' remember i am a smart toad and i shall be glad to see you',
 ' whatd i say whatd i do it myself',
 ' i was a page',
 ' jeeves was a severe trial to the third',
 ' i was a poet',
 ' i was so ashamed of my trees jack frost',
 ' i was a page',
 ' jeeves was puzzled at the recollection of the conversation',
 ' i was thinking of beginning',
 ' i was so ashamed of my trees']

In [186]:
generated_text= generate_text2(model,prime_str='i love to',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.3)
generated_text.split('.')

['i love to hunt my husband',
 ' it is a pity because i am a capital fellow',
 ' i dont know how to read it',
 ' i have never seen him before',
 ' i dont know how to give him to mine',
 ' i have been married behind',
 ' i am a traveler please excuse me',
 ' i will not tell you how to behave',
 ' but you must not talk about that',
 ' it is a pity because i am a smart toad and i dont know',
 ' i was sweeping my room my lady my lady i bought my mates in']

In [187]:
generated_text= generate_text2(model,prime_str='i love to',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.99)
generated_text.split('.')

['i love to hunt on the way',
 ' then everybody and mr',
 ' cruncher with one stone like a restless ghost',
 ' when the brothers got home he was as young as any of them returned to the town',
 ' said the princess and he gladly did his best to persuade the little princess for the stream drifted in it',
 ' the old king who had been married by a year',
 ' but his nurse had no choice but when the king spake he could not get into the outer presence of saint antoine the love of his heart and let the gardener']

In [192]:
generated_text= generate_text2(model,prime_str='i love to',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.99)
generated_text.split('.')

['i love to have a son one boy',
 ' good night good night dear me ha',
 ' forget youve got money to keep us upon em',
 ' perhaps he got up very well',
 ' i have been here for years or i have not enough to be got back again if i were to allow me',
 ' there were tables rings and thrown off',
 ' he asked',
 ' but to speak of you the books i know my work thoroughly are so wise and weak i never should find this for the expression of the house next of hers',
 ' he might become']

In [197]:
generated_text= generate_text2(model,prime_str='i want',
                            word2indx=word2indx,
                            indx2word=indx2word,
                            seq_length=100,
                            temperature=0.99)
generated_text.split('.')

['i want to go home',
 ' what did it matter when the prince did not laugh but there was silence on the beach',
 ' it had not lasted the mischief for the first time and now he would have it many useful as before',
 ' but why he dared have been ruined and the people of the world',
 ' as the rest were now close on',
 ' but the little rabbit saw the little girl was almost screamed at her to be severely',
 ' nina alexandrovna trembled and one day sara climbed down',
 ' so with a quick pause as she drew']

In [198]:
# Save the trained model and its parameters
model_save_path = "wordgen_rnn_model.pth"
params_save_path = "wordgen_rnn_params.npz"

# Save model state_dict
torch.save(model.state_dict(), model_save_path)

# Save model parameters and vocabulary
np.savez(params_save_path,
		 vocab_size=vocab_size,
		 embed_dim=embed_dim,
		 rnn_hidden_size=rnn_hidden_size,
		 word2indx=word2indx,
		 indx2word=indx2word)

In [199]:
# Save the entire trained model (architecture + weights)
full_model_save_path = "wordgen_rnn_full_model.pt"
torch.save(model, full_model_save_path)